# Procrastination patterns in Sticky Board task data

**Author:** Sajid Muhammad · B.Sc. Data Science, University of Manitoba

Sticky Board is a sticky-note task app for students. Every task records its priority, category, size, whether it was broken into steps, how many times it was snoozed, and when it was finished. This notebook uses that data to answer four questions:

1. **Which kinds of tasks get put off the most?**
2. **Do tasks that are broken into steps get snoozed less?**
3. **How long do tasks take, and how often are they finished before the first reminder?**
4. **Can we predict which tasks will be snoozed?**

> **About the data.** By default this notebook reads `data/sample_tasks_synthetic.csv`, made by `generate_sample_data.py`. That data is **synthetic**. The generator plants a few effects on purpose, for example that breaking a task into steps lowers the chance it gets snoozed. Running the notebook on it checks that the analysis can find effects that are really there. It is *not* a finding about real students.
>
> To analyse your own tasks, press **Export data (CSV)** in the app, save the file in `data/`, and change `DATA_PATH` below.

## 1. Setup

In [ ]:
from pathlib import Path
import sqlite3

import matplotlib.pyplot as plt
import pandas as pd

import sticky_analysis as sa

DATA_PATH = Path("../data/sample_tasks_synthetic.csv")   # change this to your own export
FIG_DIR = Path("figures")
SEED = 2026                                              # makes the bootstrap and CV repeatable

sa.use_chart_style()
pd.set_option("display.precision", 2)
print(f"pandas {pd.__version__} · reading {DATA_PATH}")

## 2. Load and check the data

The app exports one row per task. Times are stored in UTC; `load_tasks` converts them to Winnipeg time so that hours and weekdays line up with the student's actual day.

In [ ]:
raw = sa.load_tasks(DATA_PATH)
print(f"{len(raw)} tasks, {raw.shape[1]} columns")
raw.head()

In [ ]:
# Data-quality checks: anything listed here should be fixed or explained before trusting the results
problems = sa.check_data(raw)
print("No problems found." if not problems else "\n".join(problems))
raw.isna().sum().to_frame("missing values").T

Missing `remind_at` means the task had no reminder, and missing `done_at` means it isn't finished yet. Both are expected, not errors.

Next, add the derived columns used in the rest of the notebook: `was_snoozed`, `has_steps`, `hours_to_done`, `beat_reminder` (finished before the first reminder), `created_hour` and `created_weekday`.

In [ ]:
tasks = sa.add_features(raw)
tasks[["snoozes", "steps_total", "focus_minutes", "hours_to_done"]].describe().T

## 3. Overview

In [ ]:
# "Today" for the streak = the last moment in the data (the export time)
as_of = pd.concat([tasks["created_at"], tasks["done_at"]]).max()
finished = tasks[tasks["done"]]

overview = pd.Series({
    "tasks": len(tasks),
    "finished": f"{tasks['done'].mean():.0%}",
    "snoozed at least once": f"{tasks['was_snoozed'].mean():.0%}",
    "finished before first reminder": f"{tasks['beat_reminder'].mean():.0%}",
    "median hours to finish": f"{finished['hours_to_done'].median():.1f}",
    "focus time logged (hours)": f"{tasks['focus_minutes'].sum() / 60:.1f}",
    "streak at export (days)": sa.current_streak(tasks, today=as_of),
})
overview.to_frame("value")

## 4. Which kinds of tasks get put off the most?

**Snoozed rate** is the share of tasks snoozed at least once. A rate is easier to compare across groups of different sizes than a raw count.

In [ ]:
by_category = sa.summary_by(tasks, "category")
by_category

In [ ]:
fig = sa.plot_rate_by(
    by_category, "snoozed_rate",
    title="Share of tasks snoozed at least once, by category",
    xlabel="tasks snoozed",
    save_to=FIG_DIR / "snoozed_by_category.png",
)
plt.show()

In [ ]:
# The same summary by priority and by task size (kept in their natural order)
display(sa.summary_by(tasks, "priority", sort_by=None))
display(sa.summary_by(tasks, "size", sort_by=None))

### The same question in SQL

The export can also be loaded into SQLite (built into Python) and queried with SQL. This query adds a split by whether the task had steps.

In [ ]:
conn = sqlite3.connect(":memory:")
sa.to_sqlite(tasks, conn)

query = """
SELECT category,
       COUNT(*)                                                   AS tasks,
       ROUND(100.0 * AVG(snoozes > 0), 1)                         AS pct_snoozed,
       ROUND(AVG(CASE WHEN steps_total > 0 THEN snoozes END), 2)  AS avg_snoozes_with_steps,
       ROUND(AVG(CASE WHEN steps_total = 0 THEN snoozes END), 2)  AS avg_snoozes_without_steps
FROM tasks
GROUP BY category
HAVING COUNT(*) >= 10
ORDER BY pct_snoozed DESC;
"""
pd.read_sql_query(query, conn)

## 5. Do tasks broken into steps get snoozed less?

The app encourages writing a first small step, an idea from research on *implementation intentions* (Gollwitzer, 1999). Here we compare the share of tasks snoozed with and without steps.

- **Test:** chi-square test of independence on the 2×2 table (has steps × was snoozed).
- **Effect size:** the difference in proportions, with a 95% percentile bootstrap confidence interval (5,000 resamples).

In [ ]:
steps = sa.steps_vs_snoozing(tasks, seed=SEED)
display(steps["table"])
print(f"Snoozed, with steps:    {steps['rate_with_steps']:.1%}  (n = {steps['n_with_steps']})")
print(f"Snoozed, without steps: {steps['rate_without_steps']:.1%}  (n = {steps['n_without_steps']})")
print(f"Difference: {steps['difference']:+.1%}, 95% bootstrap CI [{steps['ci_low']:+.1%}, {steps['ci_high']:+.1%}]")
print(f"Chi-square = {steps['chi2']:.2f}, p = {steps['p_value']:.3g}")

In [ ]:
fig = sa.plot_steps_comparison(steps, save_to=FIG_DIR / "steps_vs_snoozing.png")
plt.show()

**Interpreting this carefully.** This is observational data. People choose which tasks to break down, and bigger tasks are both more likely to get steps and more likely to be snoozed. So a difference here is an *association*, not proof that adding steps causes less snoozing. The regression in section 7 adjusts for priority and size, which helps, but it can't remove choices we didn't measure.

## 6. How long do tasks take?

In [ ]:
fig = sa.plot_hours_to_done(tasks, save_to=FIG_DIR / "hours_to_done.png")
plt.show()

In [ ]:
# Hours to finish by task size: quartiles are more honest than the mean for skewed times
(tasks.groupby("size", observed=True)["hours_to_done"]
      .describe()[["count", "25%", "50%", "75%"]]
      .round(1))

In [ ]:
# How often each priority level is finished before its first reminder
(tasks.groupby("priority", observed=True)["beat_reminder"]
      .mean()
      .round(2)
      .to_frame("finished before first reminder"))

## 7. Can we predict which tasks will be snoozed?

The same question, looked at in two ways:

- **Inference (statsmodels).** A logistic regression of *was snoozed* on priority, size and steps, reported as odds ratios with 95% confidence intervals. The reference levels are normal priority, a short task, and no steps. An odds ratio below 1 means less likely to be snoozed.
- **Prediction (scikit-learn).** The same kind of model, now also using category and the hour the task was written, scored by 5-fold cross-validated ROC AUC. It's compared with a baseline that always predicts the overall snooze rate (AUC 0.5).

In [ ]:
odds = sa.snooze_odds_ratios(tasks)
odds

In [ ]:
fig = sa.plot_odds_ratios(odds, save_to=FIG_DIR / "snooze_odds_ratios.png")
plt.show()

In [ ]:
cv = sa.snooze_model_cv(tasks, folds=5, seed=SEED)
print(f"Logistic regression ROC AUC: {cv['auc_mean']:.3f} ± {cv['auc_std']:.3f}   folds: {cv['fold_auc']}")
print(f"Baseline ROC AUC:            {cv['baseline_auc']:.3f}")

An AUC of 0.5 is no better than guessing, and 1.0 would be perfect. A model somewhere in between means the task's features carry some signal. Most of what makes someone snooze a task (sleep, mood, how busy that week is) isn't in this data, so a modest AUC is the realistic result.

## 8. Daily activity

In [ ]:
daily = sa.daily_completions(tasks, days=28, end=as_of)
fig = sa.plot_daily(daily, title="Tasks finished per day, last 28 days", save_to=FIG_DIR / "tasks_per_day.png")
plt.show()

## 9. Summary and limitations

**What to look for.** The synthetic generator plants these effects. If the pipeline works, sections 4 to 7 should recover them:

- exam prep and project tasks have the highest snoozed rates (section 4)
- tasks with steps are snoozed less, with a confidence interval below zero (section 5)
- 2-minute tasks are snoozed far less than deep-work tasks, and soft-priority tasks more than urgent ones (odds ratios in section 7)

When you run this on your own export, replace this section with what you actually find.

**Limitations**

- One person's tasks make a small sample, and the same person's tasks aren't independent of each other.
- A snooze is a click in the app. It's a stand-in for procrastination, not a measure of it.
- The data is observational, so associations shouldn't be read as causes (see section 5).
- `created_hour` enters the prediction model as a straight line. A cyclic encoding or hour bins would be a natural next step.

**Next steps**

- Collect exports from classmates (with consent) and fit a mixed-effects model with a random effect per student.
- Look at workload over time: does snoozing rise in midterm weeks?

In [ ]:
conn.close()